# ML-10 — Content Action Playbook

This notebook turns validated model output into a practical, human-reviewed content action playbook.

**Decision-support only:** the score prioritizes human review. It does not automatically publish, delete, rewrite, redirect, or refresh content.

## 1. Ranked actions + reason codes

Higher score = earlier review, not certainty of decline.

Reason codes: `stale_and_visible` → refresh review; `stale_but_low_visibility` → diagnostic review; `fresh_but_high_risk` → content diagnostic; `recent_monitor` → monitor.

Archetype → action mapping: stale+visible → refresh_review; stale+low visibility → diagnostic_review; fresh+high score → content_diagnostic; recent+low score → monitor.

In [ ]:
from pathlib import Path
import json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

RANDOM_STATE = 42
repo = Path('/content/ml-internship-2026')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/engyusufayman06/ml-internship-2026.git',str(repo)], check=True)
data_path = repo / 'data/raw/content_refresh_anonymized.csv'
if not data_path.exists():
    raise FileNotFoundError('Dataset not found. Add approved FlyRank data to Colab; use HF_TOKEN through Colab Secrets.')
df = pd.read_csv(data_path)
target = 'is_declining_label'
if target not in df.columns:
    df[target] = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)
    print('Evaluation-only proxy label used.')
forbidden = {target,'trend_direction','trend_pct','content_id','client_id','score','reason_code','action_label','freshness_bucket','volume_bucket'}
features = [c for c in df.columns if c not in forbidden]
X, y = df[features].copy(), df[target].astype(int)
groups = df['client_id'].astype(str)
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X.columns if c not in num]
prep = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler())
    ]), num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), cat)
])
model = Pipeline([
    ('prep', prep),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
])
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
tr, te = next(gss.split(X, y, groups=groups))
model.fit(X.iloc[tr], y.iloc[tr])
prob = model.predict_proba(X)[:, 1]
test_prob = prob[te]
pred = (test_prob >= 0.50).astype(int)
yt = y.iloc[te].to_numpy()
metrics = {'split':'client_grouped_80_20','random_state':RANDOM_STATE,'recall':float(recall_score(yt,pred,zero_division=0)),'precision':float(precision_score(yt,pred,zero_division=0)),'f1':float(f1_score(yt,pred,zero_division=0)),'roc_auc':float(roc_auc_score(yt,test_prob))}
print(json.dumps(metrics, indent=2))
assert len(set(groups.iloc[tr]) & set(groups.iloc[te])) == 0


In [ ]:
df['review_score'] = prob
def reason(r):
    stale = r['days_since_last_update'] >= 180
    visible = r['impressions_90d'] >= 3000
    if stale and visible: return 'stale_and_visible'
    if stale: return 'stale_but_low_visibility'
    if r['review_score'] >= 0.50: return 'fresh_but_high_risk'
    return 'recent_monitor'
action = {'stale_and_visible':'refresh_review','stale_but_low_visibility':'diagnostic_review','fresh_but_high_risk':'content_diagnostic','recent_monitor':'monitor'}
df['reason_code'] = df.apply(reason, axis=1)
df['recommended_action'] = df['reason_code'].map(action)
cols = ['content_id','client_id','review_score','reason_code','recommended_action','days_since_last_update','impressions_90d']
queue = df[cols].sort_values(['review_score','impressions_90d'], ascending=[False,False]).reset_index(drop=True)
queue['priority_rank'] = np.arange(1, len(queue)+1)
display(queue.head(20))


## 2. Intended use and limits

Use the queue to help a content strategist/SEO operator decide which pages deserve human review first when capacity is limited.

Limits: grouped validation reduces client leakage but does not prove future-time performance. Selection effects and client/site mix remain possible. Cross-sectional evidence does not show that refreshing causes growth. Scores can drift as tracking, search demand, content mix, or client populations change. Low-volume items need extra caution.

Cost/value is a prioritization heuristic, not measured ROI. When capacity is scarce, high-score, high-visibility, stale pages can be reviewed first.

## 3. Human review + the no-go list

Before any action, a human checks search intent, factual accuracy, recent business/site context, evidence volume, strategic value, and the smallest safe change.

**Never automate:** publish/unpublish; rewrite/delete; URL, canonical or redirect changes; structured-data changes; medical/legal/financial or other high-stakes decisions; causal attribution; decisions based only on tiny samples; exposure of client names or private queries.

The model is a triage assistant. A human owns the final action.

## 4. Monitoring / retrain triggers

Monitor queue size, score distribution, reason-code mix, input missingness/invalid values, and sampled human-review usefulness.

Revalidate/retrain when data definitions or label logic change, new client populations arrive, score distributions shift materially, review usefulness degrades, or enough new labeled outcomes accumulate for fresh grouped/time validation. A trigger means investigate and validate again; it does not guarantee retraining improves results.

In [ ]:
monitor = {'queue_size':int(len(queue)),'score_median':float(queue.review_score.median()),'score_p90':float(queue.review_score.quantile(0.90)),'reason_code_mix':queue.reason_code.value_counts(normalize=True).round(4).to_dict()}
print(json.dumps(monitor, indent=2))


## 5. Decay / refresh insight

Earlier analysis observed that growing pages were younger on average than declining pages and reported a freshness-window difference. The practical interpretation is: **freshness is a review signal, not a causal treatment.** Older visible pages can be reasonable review candidates, but age alone should never trigger an automatic refresh. Small freshness buckets require caution.

In [ ]:
bins = [-1,30,90,180,365,np.inf]
labels = ['0–30d','31–90d','91–180d','181–365d','365+d']
tmp = df.copy()
tmp['freshness_bucket'] = pd.cut(tmp['days_since_last_update'], bins=bins, labels=labels)
decay = tmp.groupby('freshness_bucket', observed=False).agg(pages=('content_id','size'), declining_rate=(target,'mean')).reset_index()
display(decay.assign(declining_rate_pct=decay.declining_rate*100))
fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(decay.freshness_bucket.astype(str), decay.declining_rate*100, marker='o')
ax.set_xlabel('Days since last update')
ax.set_ylabel('Observed declining-label rate (%)')
ax.set_title('Observed decline rate by freshness bucket')
ax.grid(alpha=0.25)
fig.tight_layout()
fig_dir = repo/'work/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir/'w07_freshness_decay.png', dpi=160, bbox_inches='tight')
plt.show()


## 6. Exports for the paper

The queue CSV is written to `work/outputs/` and stays outside git under the repository data-leak guard. The metrics JSON is a paper receipt and should remain committed. No client names, private queries, or raw sensitive text are exported.

In [ ]:
out = repo/'work/outputs'
out.mkdir(parents=True, exist_ok=True)
queue_path = out/'w07_ranked_action_queue.csv'
metrics_path = out/'w07_playbook_metrics.json'
queue.to_csv(queue_path, index=False)
payload = {'assignment':'ML-10','notebook':'work/notebooks/w07_action_playbook.ipynb','validation':metrics,'queue_rows':int(len(queue)),'reason_code_counts':queue.reason_code.value_counts().to_dict(),'recommended_action_counts':queue.recommended_action.value_counts().to_dict(),'interpretation':'Review-priority decision support; no causal or automatic-action claim.'}
metrics_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print('Queue:', queue_path)
print('Metrics:', metrics_path)


## Self-check

- [x] Ranked actions + reason codes
- [x] Archetype → action mapping
- [x] Intended use + limits
- [x] Human review + no-go list
- [x] Monitoring / retrain triggers
- [x] Cost/value thinking
- [x] Decay/refresh insight framed as association, not causality
- [x] Queue + metrics exports
- [x] No client names, URLs, private queries, or raw sensitive content
- [ ] Run Runtime → Run all in Colab with approved data
- [ ] Save/commit the executed notebook
